# The `building_profile` agent tool

*Executed 2026-11-22 against the live Watchline graph (PLUTO 26v1, HPD registrations through 2026-05-29) and Geosupport 25b.
Re-run with `make notebook-check`; results change when the data is reloaded.*

Type a New York City address and get back the tax lot (BBL), the buildings (BIN) HPD lists for it, lot characteristics, and
counts of violations, complaint problems and litigation per building, **with the caveats that go with them**.

This notebook calls the tool exactly as the agent will: `tool.invoke({...})`, and shows what the model sees. Nothing here
is a finding about a landlord. Violations, problems and litigation are what was **cited, reported or brought**.

## Setup

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase, RoutingControl

from watchline.agent.tools import make_building_profile_tool
from watchline.geosupport.geosupport import Client
from watchline.geosupport.resolution import Neo4jGraph
from watchline.profile.profile import Neo4jProfileGraph

load_dotenv(find_dotenv(usecwd=True))          # credentials come from .env and are never printed
db = os.environ["NEO4J_DATABASE"]
driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
tool = make_building_profile_tool(Client(), Neo4jGraph(driver, db), Neo4jProfileGraph(driver, db))
print(tool.name)

building_profile


### What the model is told

The tool's description is the model's only instruction for when and how to call it.

In [2]:
print(tool.description)

Look up a New York City street address: its tax lot (BBL), the buildings (BIN) HPD lists for it, lot
characteristics, and counts of violations, complaint problems and litigation per building, with caveats. Call this first for any question that starts from a street address. The result is source-reported data:
violations, problems and litigation are what was cited, reported or brought, not findings of fault. Always
pass the returned caveats on to the user. Check status first: ok means a lookup was made (read
resolution.outcome for exact, variant, ambiguous or not_found); needs_borough means ask the user for the
borough or ZIP; invalid_input means the text was not an address; error means an outage or fault, not an
answer about the address.


A small helper prints the parts that matter. Counts are always shown **by class or type**; the tool never adds them into one number.

In [3]:
def show(out, buildings=True):
    res = out.get("resolution", {})
    print(f"status: {out['status']}" + (f"   outcome: {res['outcome']}   bbl: {res['bbl']}" if res else ""))
    if "message" in out:
        print("message:", out["message"])
    if res.get("candidates"):
        print("candidates:", res["candidates"][:5])
    if out.get("lot"):
        counts = {c["name"]: c["value"] for c in out["counts"]}
        print("counts:", counts)
    if buildings:
        for b in out.get("buildings", []):
            flag = "  <- addressed" if b["addressed"] else ""
            print(f"  BIN {b['bin']}  {b['address']}{flag}")
            for g in ("violations", "problems", "litigation"):
                if b[g]:
                    print(f"      {g}: {[{k: v for k, v in r.items()} for r in b[g]]}")
            print(f"      harassment findings (After Inquest or After Trial): {b['harassment_findings_after_inquest_or_trial']}")
    print("\ncaveats:")
    for c in out.get("caveats", []):
        print(f" - [{c['code']}] {c['text']}")

## 1. A lot with three buildings

`224 54 Street, Brooklyn` is one tax lot with three HPD-registered buildings. The tool lists all three and marks the one that was typed.
The full output is shown once, so you can see the shape the model receives.

In [4]:
d = tool.invoke({"address": "224 54 Street, Brooklyn"})
print(json.dumps(d, indent=4, sort_keys=True)[:6000])

{
    "buildings": [
        {
            "address": "224 54TH STREET",
            "addressed": true,
            "bin": "3324451",
            "harassment_findings_after_inquest_or_trial": 0,
            "litigation": [],
            "problems": [
                {
                    "duplicate": false,
                    "majorCategory": "DOOR/WINDOW",
                    "n": 3
                },
                {
                    "duplicate": false,
                    "majorCategory": "FLOORING/STAIRS",
                    "n": 1
                },
                {
                    "duplicate": false,
                    "majorCategory": "HEAT/HOT WATER",
                    "n": 3
                },
                {
                    "duplicate": false,
                    "majorCategory": "PAINT/PLASTER",
                    "n": 1
                },
                {
                    "duplicate": false,
                    "majorCategory": "UNSANITARY CONDITION

Read it in this order: `status`, then `resolution.outcome` (here `exact`), then `lot` and `counts`, then `buildings`, then `caveats`.

- **Lot-wide values** (`unitsRes` 18, `assessedTotal`, `yearBuilt`, `buildingClass`) live under `lot` and `characteristics`. They are never repeated on a building.
- **`counts`** keeps four numbers apart, each with its unit and source: parcels (1), buildings as PLUTO counts them (3), BINs HPD lists (3), residential units (18, lot-wide).
- **`totals_across_all_listed_buildings`** sums only counts of the *same* class or type across buildings.

In [5]:
show(d)

status: ok   outcome: exact   bbl: 3008210012
counts: {'parcels': 1, 'buildings (PLUTO)': 3, 'buildings (HPD)': 3, 'residential units': 18}
  BIN 3324451  224 54TH STREET  <- addressed
      problems: [{'majorCategory': 'DOOR/WINDOW', 'duplicate': False, 'n': 3}, {'majorCategory': 'FLOORING/STAIRS', 'duplicate': False, 'n': 1}, {'majorCategory': 'HEAT/HOT WATER', 'duplicate': False, 'n': 3}, {'majorCategory': 'PAINT/PLASTER', 'duplicate': False, 'n': 1}, {'majorCategory': 'UNSANITARY CONDITION', 'duplicate': False, 'n': 1}, {'majorCategory': 'WATER LEAK', 'duplicate': False, 'n': 1}]
      harassment findings (After Inquest or After Trial): 0
  BIN 3324448  218 54TH STREET
      harassment findings (After Inquest or After Trial): 0
  BIN 3324450  220 54TH STREET
      violations: [{'violationClass': 'A', 'status': 'Close', 'n': 1}, {'violationClass': 'B', 'status': 'Close', 'n': 1}, {'violationClass': 'C', 'status': 'Close', 'n': 1}]
      problems: [{'majorCategory': 'DOOR/WINDOW', 'd

## 2. A respelled street is a variant

`224 54th St` is not how the city spells the street. Geosupport matches it anyway; the tool says so and gives both spellings.

In [6]:
show(tool.invoke({"address": "224 54th St, Brooklyn"}), buildings=False)

status: ok   outcome: variant   bbl: 3008210012
counts: {'parcels': 1, 'buildings (PLUTO)': 3, 'buildings (HPD)': 3, 'residential units': 18}

caveats:
 - [variant_match] The address matched by a variant of what was typed. Typed: 224 54th St. Returned: 224 54 STREET (Geosupport return code 00).
 - [cited_not_fault] Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
 - [lot_wide] Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
 - [duplicate_flag_split] Problem counts are split by the duplicate flag; rows flagged as duplicates are counted separately.


## 3. Two building counts that disagree

PLUTO counts two buildings at `17 Battery Place`; HPD registrations list one. Neither is "the number of buildings", so both are reported, each labelled by source.

In [7]:
show(tool.invoke({"address": "17 Battery Place, Manhattan"}))

status: ok   outcome: exact   bbl: 1000157501
counts: {'parcels': 1, 'buildings (PLUTO)': 2, 'buildings (HPD)': 1, 'residential units': 500}
  BIN 1082634  1 WEST STREET  <- addressed
      violations: [{'violationClass': 'A', 'status': 'Close', 'n': 3}, {'violationClass': 'A', 'status': 'Open', 'n': 1}, {'violationClass': 'B', 'status': 'Close', 'n': 8}, {'violationClass': 'C', 'status': 'Close', 'n': 7}, {'violationClass': 'C', 'status': 'Open', 'n': 1}]
      problems: [{'majorCategory': 'DOOR/WINDOW', 'duplicate': False, 'n': 1}, {'majorCategory': 'FLOORING/STAIRS', 'duplicate': False, 'n': 2}, {'majorCategory': 'HEAT/HOT WATER', 'duplicate': False, 'n': 2}, {'majorCategory': 'PAINT/PLASTER', 'duplicate': False, 'n': 1}, {'majorCategory': 'PLUMBING', 'duplicate': False, 'n': 1}, {'majorCategory': 'UNSANITARY CONDITION', 'duplicate': False, 'n': 17}, {'majorCategory': 'WATER LEAK', 'duplicate': False, 'n': 2}]
      litigation: [{'caseType': 'Tenant Action', 'caseStatus': 'CLOSED', 

## 4. A lot with no HPD registration

`70 Broad Street` resolves to a tax lot but has no HPD-registered building. The tool says **no building recorded in the HPD registrations**, not "no buildings", and shows no violations because those exist only for registered buildings.

In [8]:
show(tool.invoke({"address": "70 Broad Street, Manhattan"}))

status: ok   outcome: exact   bbl: 1000110015
counts: {'parcels': 1, 'buildings (PLUTO)': 1, 'buildings (HPD)': 0, 'residential units': 3}

caveats:
 - [cited_not_fault] Violations, complaint problems and litigation are what was cited, reported or brought, not findings of fault. Counts are by class or type and are not a severity score.
 - [lot_wide] Residential units and assessed value are lot-wide: they belong to the lot, not to any building.
 - [no_hpd_building] No building recorded in the HPD registrations. This is not a statement that the lot has no buildings; violations, problems and litigation exist only for HPD-registered buildings.


## 5. When there is no lot to profile

Failures are answers, never exceptions, and the tool never guesses a nearby lot.

**No borough or ZIP.** Geosupport needs one. The tool asks.

In [9]:
show(tool.invoke({"address": "224 54 Street"}))

status: needs_borough
message: '224 54 Street' names no borough or ZIP. Ask the user for the borough or ZIP; do not guess.

caveats:


**An ambiguous street.** Several streets match `20 Smith`. The candidates come back and no lot is chosen.

In [10]:
show(tool.invoke({"address": "20 Smith, Brooklyn"}))

status: ok   outcome: ambiguous   bbl: None
candidates: ['SMITH JR MEM WAY', 'SMITH STREET', 'SMITHS LANE']

caveats:
 - [ambiguous_street] The street is ambiguous. Similar street names are listed as candidates; no lot was chosen.


**Not an address.**

In [11]:
show(tool.invoke({"address": "the building next to the park", "borough": "Queens"}))

status: invalid_input
message: no house number and street in 'the building next to the park'

caveats:


## 6. Litigation and the harassment finding

`21-08 Nameoke Avenue, Queens` has litigation. Look at what is counted and how it is named.

In [12]:
n = tool.invoke({"address": "21-08 Nameoke Avenue, Queens"})
show(n)

status: ok   outcome: exact   bbl: 4155250040
counts: {'parcels': 1, 'buildings (PLUTO)': 1, 'buildings (HPD)': 1, 'residential units': 4}
  BIN 4297814  21-08 NAMEOKE AVENUE  <- addressed
      violations: [{'violationClass': 'A', 'status': 'Close', 'n': 5}, {'violationClass': 'B', 'status': 'Close', 'n': 1}, {'violationClass': 'B', 'status': 'Open', 'n': 1}, {'violationClass': 'C', 'status': 'Close', 'n': 1}, {'violationClass': 'I', 'status': 'Close', 'n': 1}]
      problems: [{'majorCategory': 'APPLIANCE', 'duplicate': False, 'n': 3}, {'majorCategory': 'DOOR/WINDOW', 'duplicate': False, 'n': 4}, {'majorCategory': 'ELECTRIC', 'duplicate': False, 'n': 7}, {'majorCategory': 'FLOORING/STAIRS', 'duplicate': False, 'n': 2}, {'majorCategory': 'GENERAL', 'duplicate': False, 'n': 3}, {'majorCategory': 'HEAT/HOT WATER', 'duplicate': False, 'n': 4}, {'majorCategory': 'PAINT/PLASTER', 'duplicate': False, 'n': 8}, {'majorCategory': 'PLUMBING', 'duplicate': False, 'n': 2}, {'majorCategory': 'UNSA

The litigation rows are counts by **case type and case status**: what was brought. They are not a verdict.

Look at the numbers for this building: four cases have the type `Tenant Action/Harrassment` (spelled as recorded), yet only **one**
is counted as a harassment finding. A case type says what was *brought*, not what was *found*.

The one count with a finding in its name, `harassment_findings_after_inquest_or_trial`, includes only cases decided *After Inquest* or *After Trial*.
Across the whole graph the recorded harassment finding on a case is one of four things; the tool counts only the first two:

In [13]:
rows, _, _ = driver.execute_query(
    "CYPHER 25 MATCH (t:Litigation) RETURN coalesce(t.harassmentFinding, '(no finding recorded)') AS finding, count(*) AS cases ORDER BY cases DESC",
    database_=db, routing_=RoutingControl.READ)
for r in rows:
    print(f"{r['cases']:>8,}  {r['finding']}")

 203,962  (no finding recorded)
   5,990  No Harassment
     167  After Inquest
      50  After Trial


Most cases have **no finding recorded**, and `No Harassment` is a recorded finding of none. Neither is a harassment finding, and a missing finding is not evidence either way.
That is why the key carries its full name, so a `0` cannot be read as "no harassment".

## How to read the caveats

Every answer with a lot carries these, in the tool's own words:

| Caveat | Why it is there |
| --- | --- |
| Cited, reported or brought, **not findings of fault** | Violations, problems and litigation are allegations and records, not verdicts. |
| Residential units and assessed value are **lot-wide** | A lot is not a building; never copy or sum a lot-wide number across buildings. |
| **Two building counts** | PLUTO and HPD are different sources and often differ. |
| **No building recorded in the HPD registrations** | The building list is complete only for registered buildings. |
| Matched by a **variant**, or the resolution is **inferred** | The address was respelled; both spellings are shown. |
| Other lots **share the building's BIN** | For example a condominium; building counts are not additive across those lots. |
| Results **truncated**, or only N of M buildings shown | Row limits applied; totals still cover every listed building. |

When status is `error`, the lookup failed. That is an outage, not an answer about the address.

In [14]:
driver.close()